## Transformer Pre-Training

The aim is to pre-train a transformer model on raw price data to learn temporal price structure. 

In [1]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

Below, specific model parameters are defined. 
- Sequence Length to capture 4 hours of 5 minute data
- Fraction of bars masked in seach sequence (15% ~ 7 bars)
- transformer dimensions
- attention heads
- transformer encoder layers
- feedforward layer dimension

In [2]:
# config
SEQ_LEN    = 48
MASK_RATE  = 0.15
D_MODEL    = 64
N_HEADS    = 4
N_LAYERS   = 2
D_FF       = 256
DROPOUT    = 0.1
BATCH_SIZE = 256
EPOCHS     = 50
LR         = 1e-3
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# features
FEATURE_COLS = [
    'ret_1', 'ret_3', 'ret_6', 'ret_12', 'ret_18', 'ret_24',
    'atr_pct_12', 'vwap_dist_atr', 'vol_ratio_18', 'rv_12',
    'range_pct', 'body_pct', 'upper_wick_pct', 'lower_wick_pct',
    'tod_sin', 'tod_cos'
]

SYMBOLS     = ['COIN', 'NVDA', 'OKLO', 'PLTR', 'TSLA']
N_FEATURES  = len(FEATURE_COLS)
SYMBOL_EMB  = 8
INPUT_DIM   = N_FEATURES + SYMBOL_EMB

print(f"Device   : {DEVICE}")
print(f"Features : {N_FEATURES}")
print(f"Input dim: {INPUT_DIM} ({N_FEATURES} features + {SYMBOL_EMB} symbol embedding)")

Device   : cuda
Features : 16
Input dim: 24 (16 features + 8 symbol embedding)


Test dataset will not be utilized during pre-training.

In [3]:
# load price data
df = pd.read_csv('multiasset_5m_rth_features.csv')

df['timestamp']    = pd.to_datetime(df['timestamp'], utc=True)
df['ts_ny']        = df['timestamp'].dt.tz_convert('America/New_York')
df['session_date'] = df['ts_ny'].dt.date
df['date_ny']      = pd.to_datetime(df['session_date'])

# encode stock tickers
df['symbol_id'] = df['symbol'].map({s: i for i, s in enumerate(SYMBOLS)})

# split data in chrono order
sorted_dates = sorted(df['date_ny'].unique())
train_end    = sorted_dates[int(len(sorted_dates) * 0.70) - 1]
val_end      = sorted_dates[int(len(sorted_dates) * 0.85) - 1]

train_df = df[df['date_ny'] <= train_end].copy()
val_df   = df[(df['date_ny'] > train_end) & (df['date_ny'] <= val_end)].copy()
test_df  = df[df['date_ny'] > val_end].copy()

print(f"Train: {len(train_df):,} bars  ({train_df['date_ny'].min().date()} > {train_df['date_ny'].max().date()})")
print(f"Val  : {len(val_df):,} bars  ({val_df['date_ny'].min().date()} > {val_df['date_ny'].max().date()})")
print(f"Test : {len(test_df):,} bars  ({test_df['date_ny'].min().date()} > {test_df['date_ny'].max().date()})")
print(f"\nSymbol counts (train):")
print(train_df['symbol'].value_counts())

Train: 119,300 bars  (2024-02-28 > 2025-07-15)
Val  : 27,029 bars  (2025-07-16 > 2025-10-31)
Test : 25,363 bars  (2025-11-03 > 2026-02-27)

Symbol counts (train):
symbol
OKLO    25015
COIN    24915
PLTR    23167
NVDA    23126
TSLA    23077
Name: count, dtype: int64


Z-score normalization using training dataset only. Saved stats to utilize during fine-tuning.

In [4]:
# drop NaN rows
train_clean = train_df.dropna(subset=FEATURE_COLS)
val_clean   = val_df.dropna(subset=FEATURE_COLS)
test_clean  = test_df.dropna(subset=FEATURE_COLS)

# compute mean and std
feat_mean = train_clean[FEATURE_COLS].mean()
feat_std  = train_clean[FEATURE_COLS].std().replace(0, 1)

def normalize(df_in):
    out = df_in.copy()
    out[FEATURE_COLS] = (df_in[FEATURE_COLS] - feat_mean) / feat_std
    return out

# apply normalization to all datasets using training set stats
train_norm = normalize(train_clean)
val_norm   = normalize(val_clean)
test_norm  = normalize(test_clean)

# save scaler stats for fine tuning
scaler = {'mean': feat_mean, 'std': feat_std, 'feature_cols': FEATURE_COLS}
with open('feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Normalization stats (training set):")
stats = pd.DataFrame({'mean': feat_mean, 'std': feat_std})
print(stats.round(4))
print(f"\nScaler saved to feature_scaler.pkl")
print(f"\nRows after dropna: Train: {len(train_norm):,}  Val: {len(val_norm):,}  Test: {len(test_norm):,}")  

Normalization stats (training set):
                  mean     std
ret_1           0.0000  0.0094
ret_3           0.0001  0.0161
ret_6           0.0002  0.0228
ret_12          0.0003  0.0322
ret_18          0.0005  0.0393
ret_24          0.0006  0.0454
atr_pct_12      0.0063  0.0092
vwap_dist_atr   0.0485  2.8518
vol_ratio_18    1.0432  0.7614
rv_12           0.0045  0.0082
range_pct       0.0061  0.0058
body_pct        0.0031  0.0041
upper_wick_pct  0.0015  0.0021
lower_wick_pct  0.0015  0.0020
tod_sin        -0.0017  0.7071
tod_cos         0.0009  0.7071

Scaler saved to feature_scaler.pkl

Rows after dropna: Train: 119,300  Val: 27,029  Test: 25,363


Create PyTorch sequence dataset using sliding window. Extract preceding sequence of bars as a sequence. Sequences are built per stock and per session to avoid crossing session boundaries.

In [5]:
# create dataset for pytorch
class BarSequenceDataset(Dataset):
    def __init__(self, df, seq_len=SEQ_LEN):
        self.sequences  = []
        self.symbol_ids = []

        for symbol in SYMBOLS:
            sym_df = df[df['symbol'] == symbol].sort_values('ts_ny').reset_index(drop=True)

            for date, session in sym_df.groupby('date_ny'):
                session = session.reset_index(drop=True)
                if len(session) < seq_len:
                    continue

                feats     = session[FEATURE_COLS].values.astype(np.float32)
                symbol_id = int(session['symbol_id'].iloc[0])

                # sliding window across session bars
                for i in range(seq_len, len(session) + 1):
                    self.sequences.append(feats[i - seq_len : i])
                    self.symbol_ids.append(symbol_id)

        self.sequences  = np.array(self.sequences,  dtype=np.float32)
        self.symbol_ids = np.array(self.symbol_ids, dtype=np.int64)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.sequences[idx]),
            torch.tensor(self.symbol_ids[idx])
        )
        
train_dataset = BarSequenceDataset(train_norm)
val_dataset   = BarSequenceDataset(val_norm)

# create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train sequences : {len(train_dataset):,}")
print(f"Val sequences   : {len(val_dataset):,}")
print(f"\nSample shape - features: {train_dataset.sequences[0].shape}, symbol_id: {train_dataset.symbol_ids[0]}")
print(f"Train batches   : {len(train_loader):,}")

Train sequences : 46,816
Val sequences   : 10,664

Sample shape - features: (48, 16), symbol_id: 0
Train batches   : 183


The Transforner Model Architecture is defined below. 
- BarTransformerEncoder: core encoder that is saved and reused in fine tuning
- MaskedBarPretrainModel: encoder + reconstruction head only used in pre-training

In [6]:
# transformer model config

# sinusoidal encoding
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)
        
# core encoder
class BarTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.symbol_embedding = nn.Embedding(len(SYMBOLS), SYMBOL_EMB)
        self.input_proj       = nn.Linear(INPUT_DIM, D_MODEL)
        self.pos_encoding     = SinusoidalPositionalEncoding(D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=N_LAYERS, enable_nested_tensor=False)
        self.norm        = nn.LayerNorm(D_MODEL)
    def forward(self, x, symbol_id):
        sym_emb = self.symbol_embedding(symbol_id)
        sym_emb = sym_emb.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, sym_emb], dim=-1)
        x = self.input_proj(x)
        x = self.pos_encoding(x)                                       
        x = self.transformer(x)                                        
        return self.norm(x)

# pre-train wrapper - encoder and reconstruction head
class MaskedBarPretrainModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder      = BarTransformerEncoder()
        self.recon_head   = nn.Linear(D_MODEL, N_FEATURES)

    def forward(self, x, symbol_id):
        enc_out = self.encoder(x, symbol_id)
        recon   = self.recon_head(enc_out)
        return recon

# check parameters
model = MaskedBarPretrainModel().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)

print(f"Total model params  : {total_params:,}")
print(f"Encoder params      : {encoder_params:,}")
print(f"Reconstruction head : {total_params - encoder_params:,}")
print(f"\nModel moved to: {DEVICE}")

Total model params  : 102,776
Encoder params      : 101,736
Reconstruction head : 1,040

Model moved to: cuda


Mask bars in a sequence for the model to reconstruct. Loss is computed only on masked positions.

In [7]:
# masking logic
def apply_masking(x, mask_rate=MASK_RATE):
    B, L, _ = x.shape
    mask = torch.rand(B, L, device=x.device) < mask_rate
    masked_x = x.clone()
    masked_x[mask] = 0.0
    return masked_x, mask

# compute mse on masked positions
def masked_mse_loss(pred, target, mask):
    mask_expanded = mask.unsqueeze(-1).expand_as(pred)
    loss = nn.functional.mse_loss(
        pred[mask_expanded],
        target[mask_expanded],
        reduction='mean'
    )
    return loss

sample_x, sample_sid = next(iter(train_loader))
sample_x   = sample_x.to(DEVICE)
sample_sid = sample_sid.to(DEVICE)

masked_x, mask = apply_masking(sample_x)
pred = model(masked_x, sample_sid)
loss = masked_mse_loss(pred, sample_x, mask)

print(f"Input shape    : {sample_x.shape}")
print(f"Mask shape     : {mask.shape}")
print(f"Masked bars    : {mask.sum().item()} / {mask.numel()} ({mask.float().mean()*100:.1f}%)")
print(f"Pred shape     : {pred.shape}")
print(f"Sanity loss    : {loss.item():.4f}")

Input shape    : torch.Size([256, 48, 16])
Mask shape     : torch.Size([256, 48])
Masked bars    : 1767 / 12288 (14.4%)
Pred shape     : torch.Size([256, 48, 16])
Sanity loss    : 0.9393


Transformer training with early stopping. Best encoder weights are saved when val loss hits new minimum.

In [8]:
# transformer training
PATIENCE = 5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_val_loss    = float('inf')
patience_counter = 0
train_losses     = []
val_losses       = []

print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10}")

for epoch in range(1, EPOCHS + 1):

    # train
    model.train()
    epoch_train_loss = 0.0

    for x_batch, sid_batch in train_loader:
        x_batch   = x_batch.to(DEVICE)
        sid_batch = sid_batch.to(DEVICE)

        masked_x, mask = apply_masking(x_batch)
        pred            = model(masked_x, sid_batch)
        loss            = masked_mse_loss(pred, x_batch, mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_train_loss += loss.item()

    epoch_train_loss /= len(train_loader)
    train_losses.append(epoch_train_loss)

    # validate
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for x_batch, sid_batch in val_loader:
            x_batch   = x_batch.to(DEVICE)
            sid_batch = sid_batch.to(DEVICE)

            masked_x, mask = apply_masking(x_batch)
            pred            = model(masked_x, sid_batch)
            loss            = masked_mse_loss(pred, x_batch, mask)
            epoch_val_loss += loss.item()

    epoch_val_loss /= len(val_loader)
    val_losses.append(epoch_val_loss)
    scheduler.step()

    # early stopping
    is_best = epoch_val_loss < best_val_loss
    if is_best:
        best_val_loss    = epoch_val_loss
        patience_counter = 0
        torch.save(model.encoder.state_dict(), 'transformer_encoder_pretrained.pt')

    else:
        patience_counter += 1

    print(f"{epoch:>6} {epoch_train_loss:>12.4f} {epoch_val_loss:>10.4f}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} — no improvement for {PATIENCE} epochs.")
        break

print(f"\nBest val loss : {best_val_loss:.4f}")
print(f"Saved transformer_encoder_pretrained.pt")

 Epoch   Train Loss   Val Loss
     1       0.3616     0.1100
     2       0.2900     0.0969
     3       0.2736     0.0894
     4       0.2364     0.0839
     5       0.2391     0.0821
     6       0.2239     0.0800
     7       0.1853     0.0757
     8       0.2047     0.0730
     9       0.2010     0.0691
    10       0.2323     0.0696
    11       0.2251     0.0670
    12       0.1934     0.0656
    13       0.1757     0.0650
    14       0.2042     0.0646
    15       0.1763     0.0641
    16       0.1838     0.0638
    17       0.1758     0.0611
    18       0.1893     0.0628
    19       0.1722     0.0626
    20       0.1682     0.0612
    21       0.1649     0.0607
    22       0.1590     0.0640
    23       0.1685     0.0595
    24       0.1636     0.0607
    25       0.1583     0.0606
    26       0.1540     0.0594
    27       0.1691     0.0582
    28       0.1565     0.0598
    29       0.1467     0.0591
    30       0.1696     0.0587
    31       0.1414     0.0580
    32  

In [ ]:
# Visualize training loss curve
plt.plot(train_losses, label='Train Loss', color='steelblue')
plt.plot(val_losses, label='Val Loss', color='orange')
plt.axhline(
    best_val_loss,
    color='green',
    linestyle='--',
    linewidth=1,
    label=f'Best Val: {best_val_loss:.4f}'
)

plt.title('Pre-Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Masked MSE Loss')
plt.legend()
plt.show()